# 파일 파이프라인 추출 모듈 점검

합성 입력만 사용해 세 모듈의 대표 함수를 호출합니다.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
for extra in (REPO / "src", REPO / "src" / "card_benefit"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))
SCRATCH_ROOT = Path(tempfile.mkdtemp())


## 항목 손실 보고서

In [ ]:
from payment_card.file_pipeline.extraction.item_loss_report import build_report

# 합성 실행 결과와 청크 데이터를 임시 경로에 만들고 보고서 함수를 호출합니다.
item_loss_input_dir = SCRATCH_ROOT / "item_loss"
item_loss_input_dir.mkdir(exist_ok=True)
chunk_dir = SCRATCH_ROOT / "chunks"
chunk_dir.mkdir(exist_ok=True)
(chunk_dir / "sample.chunks.json").write_text(json.dumps({"chunks": [{"chunk_id": "c1", "text": "할인 5%"}]}), encoding="utf-8")
def write_run(name, benefits):
    folder = item_loss_input_dir / name
    folder.mkdir(exist_ok=True)
    row = {"chunk_id": "c1", "extraction": {"benefits": benefits, "conditions": [], "limits": [], "targets": [], "relations": []}}
    (folder / "c1.result.json").write_text(json.dumps(row), encoding="utf-8")
    return folder
sample_item = {"id": "b1", "name": "합성 혜택", "method": "DISCOUNT", "evidence": [{"text": "합성 근거"}]}
base_runs = [write_run("base_a", [sample_item]), write_run("base_b", [sample_item])]
compare_run = write_run("compare", [])
item_loss_result = build_report(base_runs, [compare_run], chunk_dir)
print(json.dumps(item_loss_result, ensure_ascii=False, indent=2))


## 합의 추출 결과

In [ ]:
from payment_card.file_pipeline.extraction.consensus import build_consensus

# 같은 합성 혜택을 두 실행에서 확인해 합의 결과를 표시합니다.
def sample_run(item_id):
    return {"chunk_id": "c2", "extraction": {"schema_ver": "ONTOLOGY_V1", "benefits": [{"id": item_id, "name": "합성 혜택", "evidence": [{"text": "합성 근거"}]}], "conditions": [], "limits": [], "targets": [], "relations": []}, "failures": [], "notes": []}
consensus_result = build_consensus([[sample_run("b1")], [sample_run("b2")]], k=2)
print(json.dumps(consensus_result, ensure_ascii=False, indent=2))


## 표 밀도 점수

In [ ]:
from payment_card.file_pipeline.extraction.table_density_report import density_score

# 경로 없이 합성 텍스트를 넣어 밀도 점수와 결과를 표시합니다.
density_example = "항목  혜택\n국내 가맹점  5% 할인\n월 한도  10,000원"
density_result = {"input": density_example, "score": density_score(density_example)}
print(json.dumps(density_result, ensure_ascii=False, indent=2))
